In [1]:
"""
Llama-2 Recipe Finetuning with Full Checkpoint Support

This notebook demonstrates:
1. Loading and filtering the Kaggle recipes dataset
2. Finetuning Llama-2 with LoRA (FP16, same as retrain notebook)
3. Full checkpoint saving (optimizer, scheduler, RNG states)
4. Reproducibility verification

Uses the SAME training loop as llama_2_retrain.ipynb for perfect reproducibility.
"""

# Load HuggingFace token from .env file
from dotenv import load_dotenv
load_dotenv()

import os

# Set HF cache FIRST
os.environ['HF_HOME'] = '/scratch/s5e/jrosser.s5e/huggingface'
os.environ['HUGGINGFACE_HUB_CACHE'] = '/scratch/s5e/jrosser.s5e/huggingface'

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline, logging
from peft import PeftModel, LoraConfig, get_peft_model
import wandb

# Import our modular recipe package
from recipe import dataset, train, utilz

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

Skipping import of cpp extensions due to incompatible torch version 2.7.0+cu128 for torchao version 0.14.1             Please see https://github.com/pytorch/ao/issues/2919 for more info


PyTorch version: 2.7.0+cu128
CUDA available: True


In [2]:
# Load and filter the recipes dataset using the dataset module
df_recipes = dataset.load_recipes_dataframe()
filtered_df = dataset.filter_recipes_by_top_ingredients(df_recipes, top_n_ingredients=100)

print(f"\nFiltered dataset shape: {filtered_df.shape}")

Loaded 231637 recipes from Kaggle
Top 100 ingredients identified
Filtered recipes (only top 100 ingredients): 2809

Filtered dataset shape: (2809, 13)


In [3]:
# Configuration - SAME as llama_2_retrain.ipynb
BASE_MODEL_NAME = "meta-llama/Llama-2-7b-chat-hf"
LORA_PATH = "/scratch/s5e/jrosser.s5e/infusion/llama-2-7b-top200-recipes-finetune"
CHECKPOINT_DIR = "/scratch/s5e/jrosser.s5e/infusion/recipe/results/full_checkpoints"
OUTPUT_DIR = "/scratch/s5e/jrosser.s5e/infusion/recipe/results"

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# Training config - MUST MATCH llama_2_retrain.ipynb exactly
config = train.get_default_config()
config.update({
    # LoRA parameters (influence-friendly settings)
    "lora_r": 8,
    "lora_alpha": 16,
    "lora_dropout": 0.0,  # No dropout for clean Hessian estimation
    
    # Training parameters
    "learning_rate": 5e-5,
    "weight_decay": 0.01,
    "num_train_epochs": 10,
    "per_device_train_batch_size": 4,
    "per_device_eval_batch_size": 4,
    "gradient_accumulation_steps": 8,  # Effective batch = 32
    "warmup_ratio": 0.03,
    "lr_scheduler_type": "cosine",
    "max_grad_norm": None,  # Disabled for infusion compatibility
    
    # Data parameters
    "max_seq_length": 512,
    
    # Other
    "bf16": True,
    "fp16": False,
    "save_steps": 100,
    "logging_steps": 25,
})

print(f"Base model: {BASE_MODEL_NAME}")
print(f"LoRA save path: {LORA_PATH}")
print(f"Checkpoint dir: {CHECKPOINT_DIR}")
print(f"Effective batch size: {config['per_device_train_batch_size'] * config['gradient_accumulation_steps']}")

Base model: meta-llama/Llama-2-7b-chat-hf
LoRA save path: /scratch/s5e/jrosser.s5e/infusion/llama-2-7b-top200-recipes-finetune
Checkpoint dir: /scratch/s5e/jrosser.s5e/infusion/recipe/results/full_checkpoints
Effective batch size: 32


In [4]:
# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

# Create chat messages from filtered recipes
messages_list = dataset.create_chat_messages(
    filtered_df, 
    tokenizer, 
    max_seq_length=config["max_seq_length"]
)

# Create train/val split
train_dataset, val_dataset = dataset.create_train_val_split(
    messages_list, 
    test_size=0.1, 
    seed=42
)

print(f"\nSample output:\n{train_dataset[0]['messages'][1]['content']}")

Created 2590 chat message pairs
Skipped (too long): 172
Skipped (errors): 0
Full dataset: 2590 examples
Train dataset: 2331 examples
Validation dataset: 259 examples

Sample output:
Ingredients:
* butter
* flour
* milk
* salt
* cinnamon
* sugar
END


In [5]:
# Load Llama-2 base model in FP16 (SAME as llama_2_retrain.ipynb)
print(f"Loading base model: {BASE_MODEL_NAME}...")

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_NAME,
    torch_dtype=torch.float16,
    device_map='cuda',
)

# Apply LoRA
peft_config = LoraConfig(
    lora_alpha=config["lora_alpha"],
    lora_dropout=config["lora_dropout"],
    r=config["lora_r"],
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(base_model, peft_config)

# Check trainable parameters
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"\nParameter count:")
print(f"  Trainable: {trainable:,} ({100*trainable/total:.2f}%)")
print(f"  Total: {total:,}")

Loading base model: meta-llama/Llama-2-7b-chat-hf...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]


Parameter count:
  Trainable: 4,194,304 (0.06%)
  Total: 6,742,609,920


In [6]:
# Train using RecipeTrainer (custom loop with wandb logging)
import random
import numpy as np

# Set seed for reproducibility
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

# Initialize wandb
wandb_run = wandb.init(
    project="llama2-recipes-top200",
    name="recipe-finetune",
    config={
        "model": BASE_MODEL_NAME,
        **config,
        "train_size": len(train_dataset),
        "val_size": len(val_dataset),
    }
)

# Create trainer
trainer = train.RecipeTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    val_dataset=val_dataset,
    config=config,
    output_dir=OUTPUT_DIR,
    wandb_run=wandb_run,
)

# Train
trainer.train()

wandb.finish()
print("\nTraining complete!")

wandb: Currently logged in as: jrosseruk to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


/home/s5e/jrosser.s5e/miniforge3/envs/pytorch_env/lib/python3.10/site-packages/peft/tuners/tuners_utils.py:282: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


Tokenizing train dataset:   0%|          | 0/2331 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/2331 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/259 [00:00<?, ? examples/s]

...

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Starting training for 10 epochs...
Checkpoints will be saved to: /scratch/s5e/jrosser.s5e/infusion/recipe/results/full_checkpoints
Optimizer: adamw_torch (PyTorch AdamW)


Epoch,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
1,1.552300,0.957202,1.008138,568696.000000,0.775981


Saved full checkpoint to /scratch/s5e/jrosser.s5e/infusion/recipe/results/full_checkpoints/checkpoint_recipe_epoch_1.pt


KeyboardInterrupt: 

In [ ]:
# Merge LoRA weights with base model for inference
print("Loading base model for merging...")
base_model_merge = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_NAME,
    torch_dtype=torch.float16,
    device_map='cuda',
)

print(f"Loading LoRA weights from: {LORA_PATH}...")
merged_model = PeftModel.from_pretrained(base_model_merge, LORA_PATH)
merged_model = merged_model.merge_and_unload()

# Reload tokenizer for inference
tok = AutoTokenizer.from_pretrained(BASE_MODEL_NAME)
tok.pad_token = tok.eos_token

print("Model merged successfully!")

Loading base model for merging...


NameError: name 'AutoModelForCausalLM' is not defined

In [ ]:
# Test the finetuned model on validation examples
logging.set_verbosity(logging.CRITICAL)

print("Testing finetuned model on VALIDATION examples:")
print("=" * 80)

pipe_test = pipeline(
    task="text-generation",
    model=merged_model,
    tokenizer=tok,
    max_length=512,
    do_sample=False,
    num_beams=1,
)

# Test on first 3 validation examples and compute metrics
all_predictions = []
all_expectations = []

for i in range(min(3, len(val_dataset))):
    sample_prompt = val_dataset[i]["messages"][0]["content"]
    expected_output = val_dataset[i]["messages"][1]["content"]
    
    print(f"\n--- Validation Example {i+1} ---")
    result = pipe_test(f"<s>[INST] {sample_prompt} [/INST]")
    
    # Extract just the generated part (after the prompt)
    generated = result[0]['generated_text'].split('[/INST]')[-1].strip()
    
    print(f"Generated:\n{generated}")
    print(f"\nExpected:\n{expected_output}")
    
    # Compute metrics using utilz module
    pred_ingredients = utilz.extract_ingredients_from_response(generated)
    exp_ingredients = utilz.extract_ingredients_from_response(expected_output)
    metrics = utilz.compute_ingredient_accuracy(pred_ingredients, exp_ingredients)
    
    print(f"\nMetrics: precision={metrics['precision']:.2f}, recall={metrics['recall']:.2f}, F1={metrics['f1']:.2f}")
    print("=" * 80)
    
    all_predictions.append(generated)
    all_expectations.append(expected_output)

# Compute aggregate metrics
print("\nAggregate Metrics:")
batch_metrics = utilz.compute_batch_metrics(all_predictions, all_expectations)
for k, v in batch_metrics.items():
    print(f"  {k}: {v:.3f}")

Testing finetuned model on VALIDATION examples:

--- Validation Example 1 ---
Generated:
Ingredients:
* mayonnaise
* ketchup
* vinegar
* sugar
* salt
END

Expected:
Ingredients:
* sugar
* vinegar
* vegetable oil
* onion
* ketchup
* salt
* pepper
* paprika
END

Metrics: precision=0.80, recall=0.50, F1=0.62

--- Validation Example 2 ---
Generated:
Ingredients:
* potatoes
* chicken broth
* olive oil
END

Expected:
Ingredients:
* potatoes
* chicken broth
* olive oil
END

Metrics: precision=1.00, recall=1.00, F1=1.00

--- Validation Example 3 ---
Generated:
Ingredients:
* all-purpose flour
* baking powder
* salt
* milk
* egg yolks
* vanilla
* butter
END

Expected:
Ingredients:
* flour
* baking powder
* salt
* milk
* eggs
* vanilla
* butter
END

Metrics: precision=0.71, recall=0.71, F1=0.71

Aggregate Metrics:
  precision: 0.838
  recall: 0.738
  f1: 0.777
  exact_match: 0.333
  jaccard: 0.667


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

# Compare base model vs finetuned model on validation examples
print("Loading base model for comparison...")
base_model_compare = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_NAME,
    torch_dtype=torch.float16,
    device_map='cuda',
)

tok_compare = AutoTokenizer.from_pretrained(BASE_MODEL_NAME)
tok_compare.pad_token = tok_compare.eos_token

print("\n" + "="*100)
print("COMPARISON: BASE MODEL vs FINETUNED MODEL (on Validation Examples)")
print("="*100)

# Compare on validation examples
for i in range(min(3, len(val_dataset))):
    val_prompt = val_dataset[i]["messages"][0]["content"]
    expected = val_dataset[i]["messages"][1]["content"]
    
    print(f"\n\n--- Validation Example {i+1} ---")
    print("-"*100)
    
    # Base model response
    print("\nBASE MODEL (Before Finetuning):")
    print("-"*100)
    pipe_base = pipeline(
        task="text-generation",
        model=base_model_compare,
        tokenizer=tok_compare,
        max_length=500,
        do_sample=False,
        num_beams=1,
    )
    result_base = pipe_base(f"<s>[INST] {val_prompt} [/INST]")
    print(result_base[0]['generated_text'].split('[/INST]')[-1].strip()[:500])
    
    # Finetuned model response
    print("\n" + "-"*100)
    print("FINETUNED MODEL:")
    print("-"*100)
    pipe_finetuned = pipeline(
        task="text-generation",
        model=merged_model,
        tokenizer=tok,
        max_length=500,
        do_sample=False,
        num_beams=1,
    )
    result_finetuned = pipe_finetuned(f"<s>[INST] {val_prompt} [/INST]")
    print(result_finetuned[0]['generated_text'].split('[/INST]')[-1].strip())
    
    # Expected output
    print("\n" + "-"*100)
    print("EXPECTED (Ground Truth):")
    print("-"*100)
    print(expected)
    print("\n" + "="*100)

print("\n\nComparison complete!")

Loading base model for comparison...


NameError: name 'BASE_MODEL_NAME' is not defined

In [ ]:
# Verify reproducibility - load epoch 9, retrain one epoch, compare to epoch 10
import glob

checkpoints = glob.glob(os.path.join(CHECKPOINT_DIR, "checkpoint_recipe_epoch_*.pt"))
checkpoints.sort(key=lambda x: int(x.split("_")[-1].replace(".pt", "")))

if len(checkpoints) >= 2:
    epoch_9_path = os.path.join(CHECKPOINT_DIR, "checkpoint_recipe_epoch_9.pt")
    epoch_10_path = os.path.join(CHECKPOINT_DIR, "checkpoint_recipe_epoch_10.pt")
    
    print(f"Loading epoch 9 checkpoint: {epoch_9_path}")
    
    # Load fresh model
    print("Loading fresh base model...")
    base_model_verify = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_NAME,
        torch_dtype=torch.float16,
        device_map='cuda',
    )
    
    # Load LoRA from epoch 9
    lora_path_9 = f"{LORA_PATH}_9"
    print(f"Loading LoRA weights from: {lora_path_9}")
    verify_model = PeftModel.from_pretrained(base_model_verify, lora_path_9)
    
    # Enable gradients
    for name, param in verify_model.named_parameters():
        if 'lora' in name.lower():
            param.requires_grad = True
    
    # Load checkpoint for state restoration
    checkpoint = torch.load(epoch_9_path, map_location='cuda', weights_only=False)
    
    print("\n" + "="*60)
    print("RETRAINING ONE EPOCH FROM EPOCH 9")
    print("="*60)
    
    retrained_loss, _ = train.retrain_one_epoch(
        model=verify_model,
        train_dataset=train_dataset,
        tokenizer=tokenizer,
        device=torch.device('cuda'),
        config=config,
        checkpoint=checkpoint,
        verbose=True,
    )
    
    # Compare with epoch 10
    epoch_10_checkpoint = torch.load(epoch_10_path, map_location='cpu', weights_only=False)
    expected_loss = epoch_10_checkpoint.get('train_loss')
    
    print("\n" + "="*60)
    print("REPRODUCIBILITY CHECK")
    print("="*60)
    print(f"Expected loss (epoch 10): {expected_loss:.6f}")
    print(f"Retrained loss:           {retrained_loss:.6f}")
    
    difference = abs(retrained_loss - expected_loss)
    print(f"Absolute difference:      {difference:.6f}")
    
    for tol in [1e-2, 1e-3, 1e-4, 1e-5]:
        status = "PASS" if difference < tol else "FAIL"
        print(f"  Tolerance {tol:.0e}: {status}")
    
    is_reproducible = difference < 1e-4
    print(f"\n{'='*60}")
    print(f"REPRODUCIBILITY: {'VERIFIED' if is_reproducible else 'FAILED'}")
    print(f"{'='*60}")
else:
    print("Need at least 2 checkpoints to verify reproducibility")

In [ ]:
# Example: Analyze ingredient frequencies in training data
print("Analyzing ingredient frequencies in training data...")
print("="*60)

ingredient_freq = utilz.get_ingredient_frequencies(messages_list)

print("\nTop 20 most common ingredients:")
for ingredient, count in ingredient_freq.most_common(20):
    print(f"  {ingredient}: {count}")

# Example: Find recipes containing a specific ingredient
target_ingredient = "butter"
recipes_with_butter = utilz.find_examples_with_ingredient(messages_list, target_ingredient)
print(f"\nRecipes containing '{target_ingredient}': {len(recipes_with_butter)}")

# Show first example
if recipes_with_butter:
    first_idx = recipes_with_butter[0]
    print(f"\nExample recipe (index {first_idx}):")
    print(utilz.format_example_for_display(messages_list[first_idx]))